In [1]:
import os
import json
import pandas as pd
import numpy as np
import cv2
from scipy.signal.windows import gaussian as scipy_gaussian

import matplotlib.pyplot as plt

In [2]:
# 1. Paramètres
CSV_PATH    = "label_annotated.csv"   # votre export CSV de VIA
IMG_DIR     = "dataset_project_iapr2025/train"                # dossier de vos .JPG
OUT_DIR     = "density_maps/"                                 # dossier où sauver les .npz
SIGMA       = 150                                             # écart-type de la gaussienne
CLASS_NAMES = [
    "Jelly_White","Jelly_Milk","Jelly_Black","Amandina","Crème_brûlée",
    "Triangolo","Tentation_noir","Comtesse","Noblesse","Noir_authentique",
    "Passion_au_lait","Arabia","Stracciatella"
]

In [16]:
# 2. Mapping nom → indice
class2idx = {name: i for i, name in enumerate(CLASS_NAMES)}

In [17]:
# 3. Précompute the Gaussian kernel
kern_size = int(6 * SIGMA) | 1   # kernel size, ensure odd
g1d = scipy_gaussian(kern_size, SIGMA)
kernel2d = np.outer(g1d, g1d)
kernel2d /= kernel2d.sum()       # normalize sum to 1
half_k = kern_size // 2

In [18]:
# 4. Charger le CSV
df = pd.read_csv(CSV_PATH)

In [19]:
# 5. Créer le dossier de sortie
os.makedirs(OUT_DIR, exist_ok=True)

In [20]:
# 6. Boucle sur chaque image
for fname, group in df.groupby("filename"):
    # 6.1 Lire l'image pour connaître ses dimensions
    img_path = os.path.join(IMG_DIR, fname)
    img = cv2.imread(img_path)
    if img is None:
        print(f"[ERROR] Impossible de lire l'image : {img_path}")
        continue
    H, W = img.shape[:2]
    
    # 6.2 Initialiser le tenseur de density maps
    density = np.zeros((H, W, len(CLASS_NAMES)), dtype=np.float32)
    
    # 6.3 Pour chaque point annoté, scatter-add le kernel
    for _, row in group.iterrows():
        shape_attr = json.loads(row["region_shape_attributes"])
        x = int(shape_attr["cx"])
        y = int(shape_attr["cy"])
        choco_type = json.loads(row["region_attributes"])["Choco_Type"]
        c = class2idx.get(choco_type, None)
        if c is None:
            continue
        
        # bounds for scatter-add
        y0 = max(0,   y - half_k)
        y1 = min(H,   y + half_k + 1)
        x0 = max(0,   x - half_k)
        x1 = min(W,   x + half_k + 1)
        ky0 = half_k - (y - y0)
        ky1 = half_k + (y1 - y)
        kx0 = half_k - (x - x0)
        kx1 = half_k + (x1 - x)
        
        density[y0:y1, x0:x1, c] += kernel2d[ky0:ky1, kx0:kx1]
    
    # 6.4 Conversion en float16 pour réduire la taille
    density = density.astype(np.float16)
    
    # 6.5 Sauvegarder au format compressé .npz
    out_path = os.path.join(OUT_DIR, fname.replace(".JPG", ".npz"))
    np.savez_compressed(out_path, density=density)
    
    print(f"Saved compressed density map for {fname} → {out_path}")


Saved compressed density map for L1000756.JPG → density_maps/L1000756.npz
Saved compressed density map for L1000763.JPG → density_maps/L1000763.npz
Saved compressed density map for L1000765.JPG → density_maps/L1000765.npz


KeyboardInterrupt: 

In [1]:
# exemple de chargement d'une density map et de l'image correspondante
npz = np.load("density_maps/L1000763.npz")
density = npz["density"]    # shape (H, W, n_classes)
img = cv2.imread("dataset_project_iapr2025/train/L1000763.JPG")[:,:,::-1]

# visualisation de la somme sur les canaux (densité totale)
total_density = density.sum(axis=-1)

fig, axes = plt.subplots(1,2, figsize=(12,5))
axes[0].imshow(img); axes[0].set_title("Image originale"); axes[0].axis("off")
axes[1].imshow(total_density, cmap="jet"); axes[1].set_title("Density map (somme)"); axes[1].axis("off")
plt.show()

# vérifiez aussi la somme pour estimer le nombre d'objets
print("Nombre estimé d'objets sur l'image :", total_density.sum())


NameError: name 'np' is not defined

# Neural network :

In [3]:
import os
import json
import random
from glob import glob

import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [4]:
# ─── 1. CONFIG ────────────────────────────────────────────────────────────────
DM_DIR      = "density_maps"
N_CLASSES = len(CLASS_NAMES)
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH     = 4
EPOCHS    = 30
LR        = 1e-4
SEED      = 42

# Réglage du seed
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [5]:
import os, random, cv2, numpy as np, torch
from torch.utils.data import Dataset

class ChocolateDataset(Dataset):
    def __init__(self, files, img_dir, dm_dir,
                 target_size=(512,512),
                 affine_params=None,
                 augment=True):
        """
        files         : liste de noms de fichiers .JPG
        img_dir, dm_dir: dossiers images / .npz
        target_size   : (H, W) taille en sortie
        affine_params : dict avec
            'translate_frac': fraction max de W/H pour la translation
            'scale_range'   : tuple (min, max) pour le zoom
            'shear_deg'     : angle max de cisaillement en degrés
        augment       : bool, si False on désactive la partie aléatoire
        """
        self.files       = files
        self.img_dir     = img_dir
        self.dm_dir      = dm_dir
        self.Th, self.Tw = target_size
        self.ap = affine_params or {
            'translate_frac':0.1,
            'scale_range':(0.9,1.1),
            'shear_deg':10
        }
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        # 1) Lecture & resize
        img = cv2.imread(os.path.join(self.img_dir, fname))
        if img is None:
            raise FileNotFoundError(f"Image not found: {fname}")
        img = img[:,:,::-1].astype(np.float32)/255.0
        H0, W0 = img.shape[:2]
        img = cv2.resize(img, (self.Tw, self.Th), interpolation=cv2.INTER_AREA)

        dm = np.load(os.path.join(self.dm_dir, fname.replace(".JPG",".npz")))["density"].astype(np.float32)
        scale_area = (H0 * W0) / (self.Th * self.Tw)
        dm_chw = dm.transpose(2,0,1)
        dm_rs = np.zeros((dm_chw.shape[0], self.Th, self.Tw), dtype=np.float32)
        for c in range(dm_chw.shape[0]):
            patch = cv2.resize(dm_chw[c], (self.Tw, self.Th), interpolation=cv2.INTER_AREA)
            dm_rs[c] = patch * scale_area

        # 2) Tirage des paramètres affine uniquement si augment=True
        if self.augment:
            tx = random.uniform(-self.ap['translate_frac'], self.ap['translate_frac']) * self.Tw
            ty = random.uniform(-self.ap['translate_frac'], self.ap['translate_frac']) * self.Th
            sc = random.uniform(*self.ap['scale_range'])
            sh = random.uniform(-self.ap['shear_deg'], self.ap['shear_deg'])
            a, d = sc, sc
            b = np.tan(np.deg2rad(sh))
            c_ = 0.0
            M = np.array([[a, b, tx],
                          [c_, d, ty]], dtype=np.float32)
            det = a * d - b * c_

            img = cv2.warpAffine(img, M, (self.Tw, self.Th),
                                 flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_REFLECT)
            for c in range(dm_rs.shape[0]):
                dm_rs[c] = cv2.warpAffine(dm_rs[c], M, (self.Tw, self.Th),
                                          flags=cv2.INTER_LINEAR,
                                          borderMode=cv2.BORDER_REFLECT) * det

        # 3) Conversion en Tensors
        img_t = torch.from_numpy(img.transpose(2,0,1)).float()   # (3,H,W)
        dm_t  = torch.from_numpy(dm_rs).float()                   # (C,H,W)

        return img_t, dm_t




from glob import glob
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

# 1) Récupère tous tes fichiers .JPG
all_files = [os.path.basename(p) for p in glob(os.path.join(IMG_DIR, "*.JPG"))]

# 2) Split train / val
train_files, val_files = train_test_split(all_files, test_size=0.2, random_state=42)

# 3) Datasets
train_ds = ChocolateDataset(
    files=train_files,
    img_dir=IMG_DIR,
    dm_dir=DM_DIR,
    target_size=(512,512),
    affine_params={'translate_frac':0.1,'scale_range':(0.9,1.1),'shear_deg':10},
    augment=True
)
val_ds = ChocolateDataset(
    files=val_files,
    img_dir=IMG_DIR,
    dm_dir=DM_DIR,
    target_size=(512,512),
    affine_params={'translate_frac':0.1,'scale_range':(0.9,1.1),'shear_deg':10},
    augment=False  # pas d’augment sur la val
)

# 4) DataLoaders
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=2, shuffle=False, num_workers=0, pin_memory=False)

# 5) Sanity-check
imgs, dms = next(iter(train_loader))
print("Train batch  →", imgs.shape, dms.shape)
imgs, dms = next(iter(val_loader))
print("Val batch    →", imgs.shape, dms.shape)

Train batch  → torch.Size([2, 3, 512, 512]) torch.Size([2, 13, 512, 512])
Val batch    → torch.Size([2, 3, 512, 512]) torch.Size([2, 13, 512, 512])


In [6]:
# ─── 3. UNet SIMPLE ─────────────────────────────────────────────────────────────
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.down1 = DoubleConv(3,  64)
        self.pool  = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128,256)
        self.down4 = DoubleConv(256,512)

        self.up3 = nn.ConvTranspose2d(512,256,2,stride=2)
        self.conv3 = DoubleConv(512,256)
        self.up2 = nn.ConvTranspose2d(256,128,2,stride=2)
        self.conv2 = DoubleConv(256,128)
        self.up1 = nn.ConvTranspose2d(128,64,2,stride=2)
        self.conv1 = DoubleConv(128,64)

        self.final = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        d4 = self.down4(self.pool(d3))

        u3 = self.up3(d4)
        u3 = torch.cat([u3, d3], dim=1)
        u3 = self.conv3(u3)

        u2 = self.up2(u3)
        u2 = torch.cat([u2, d2], dim=1)
        u2 = self.conv2(u2)

        u1 = self.up1(u2)
        u1 = torch.cat([u1, d1], dim=1)
        u1 = self.conv1(u1)

        return self.final(u1)

# Instanciation
model = UNet(N_CLASSES).to(DEVICE)

In [7]:
from torchsummary import summary
summary(model, (3, 512, 512), batch_size=1)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [1, 64, 512, 512]           1,728
       BatchNorm2d-2          [1, 64, 512, 512]             128
              ReLU-3          [1, 64, 512, 512]               0
            Conv2d-4          [1, 64, 512, 512]          36,864
       BatchNorm2d-5          [1, 64, 512, 512]             128
              ReLU-6          [1, 64, 512, 512]               0
        DoubleConv-7          [1, 64, 512, 512]               0
         MaxPool2d-8          [1, 64, 256, 256]               0
            Conv2d-9         [1, 128, 256, 256]          73,728
      BatchNorm2d-10         [1, 128, 256, 256]             256
             ReLU-11         [1, 128, 256, 256]               0
           Conv2d-12         [1, 128, 256, 256]         147,456
      BatchNorm2d-13         [1, 128, 256, 256]             256
             ReLU-14         [1, 128, 2

In [8]:
# ─── 4. LOSS, OPTI, SCHEDULER ───────────────────────────────────────────────────
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
                                                       mode='min',
                                                       factor=0.5,
                                                       patience=3,
                                                       verbose=True)

/opt/jlab-env/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [9]:
# ─── 5. FONCTIONS D’ENTRAÎNEMENT & ÉVAL ─────────────────────────────────────────
def train_epoch(loader, model, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for imgs, dms in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(DEVICE)
        dms  = dms.to(DEVICE)
        preds = model(imgs)
        loss  = criterion(preds, dms)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    return running_loss / len(loader.dataset)


from sklearn.metrics import f1_score
def eval_epoch(loader, model, criterion):
    model.eval()
    running_loss = 0.0

    # Pour accumuler les vrais/pred pour tout le split
    all_true_counts = []
    all_pred_counts = []
    mae_per_class = torch.zeros(N_CLASSES, device=DEVICE)

    with torch.no_grad():
        for imgs, dms in loader:
            imgs = imgs.to(DEVICE)
            dms  = dms.to(DEVICE)

            # 1) Forward + loss
            preds = model(imgs)
            running_loss += criterion(preds, dms).item() * imgs.size(0)

            pred_cnt = (preds.sum(dim=(2,3))        # (B, C) float
                             .clamp(min=0)          # plus jamais négatif
                             .round()               # arrondi à l’entier le plus proche
                             .long()                # tensor int64
                             .cpu()
                             .numpy())             # (B, C) int

            true_cnt = (dms.sum(dim=(2,3))           # (B, C) float
                           .round()                  # si votre ground-truth était float
                           .long()
                           .cpu()
                           .numpy())                # (B, C) int

            # 3) MAE
            mae_per_class += torch.from_numpy(
                np.abs(pred_cnt - true_cnt)
            ).sum(dim=0).to(DEVICE)

            # 4) Stocker pour le F1
            all_true_counts.append(true_cnt)
            all_pred_counts.append(pred_cnt)

    # Concaténer tout
    all_true = np.vstack(all_true_counts)   # shape (N_images, C)
    all_pred = np.vstack(all_pred_counts)

    # Binarisation : présence (1) si count>0, sinon absence (0)
    y_true = (all_true > 0).astype(int)
    y_pred = (all_pred > 0).astype(int)

    # F1 par classe et F1 macro (moyenne non pondérée)
    f1_per_class = f1_score(y_true, y_pred, average=None)    # array de taille C
    f1_macro     = f1_score(y_true, y_pred, average='macro') # scalaire

    # Moyenne MAE par classe
    mae = (mae_per_class / len(loader.dataset)).cpu().numpy()

    # Loss moyenne
    loss = running_loss / len(loader.dataset)

    return loss, mae, f1_per_class, f1_macro

In [ ]:
# ─── 6. BOUCLE PRINCIPALE ───────────────────────────────────────────────────────
best_val_loss = float('inf')
for epoch in range(1, EPOCHS+1):
    tr_loss = train_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_mae, val_f1_pc, val_f1_macro = eval_epoch(val_loader, model, criterion)
    scheduler.step(val_loss)

    print(f"Époch {epoch:02d} | "
          f"Train Loss: {tr_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"MAE: {val_mae.mean():.3f} | "
          f"F1_macro: {val_f1_macro:.3f}")

    # optionnel : afficher le F1 par classe
    for cname, f1 in zip(CLASS_NAMES, val_f1_pc):
        print(f"  F1 {cname:<15s}: {f1:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_choco_unet.pth")
        print("→ Modèle sauvegardé!")

Époch 01 | Train Loss: 0.0450 | Val Loss: 0.0086 | MAE: 7688.483 | F1_macro: 0.228
  F1 Jelly_White    : 0.000
  F1 Jelly_Milk     : 0.444
  F1 Jelly_Black    : 0.500
  F1 Amandina       : 0.400
  F1 Crème_brûlée   : 0.000
  F1 Triangolo      : 0.000
  F1 Tentation_noir : 0.500
  F1 Comtesse       : 0.375
  F1 Noblesse       : 0.000
  F1 Noir_authentique: 0.000
  F1 Passion_au_lait: 0.500
  F1 Arabia         : 0.000
  F1 Stracciatella  : 0.250
→ Modèle sauvegardé!


Époch 02 | Train Loss: 0.0055 | Val Loss: 0.0030 | MAE: 2495.624 | F1_macro: 0.262
  F1 Jelly_White    : 0.000
  F1 Jelly_Milk     : 0.308
  F1 Jelly_Black    : 0.381
  F1 Amandina       : 0.375
  F1 Crème_brûlée   : 0.000
  F1 Triangolo      : 0.000
  F1 Tentation_noir : 0.500
  F1 Comtesse       : 0.522
  F1 Noblesse       : 0.286
  F1 Noir_authentique: 0.286
  F1 Passion_au_lait: 0.455
  F1 Arabia         : 0.167
  F1 Stracciatella  : 0.125
→ Modèle sauvegardé!


Train:  22%|██▏       | 8/36 [00:56<03:17,  7.04s/it]

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# 1) Vos définitions existantes :
#    - UNet (ou votre classe de modèle)
#    - DEVICE (ex. torch.device("cuda" if torch.cuda.is_available() else "cpu"))
#    - CLASS_NAMES : liste de vos 13 noms de classes, dans l'ordre.

# 2) Dataset test (images seules)
class TestImageDataset(Dataset):
    def __init__(self, img_dir, file_list, target_size=(512,512)):
        self.img_dir = img_dir
        self.files = file_list
        self.Th, self.Tw = target_size

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        path = os.path.join(self.img_dir, fname)
        img = cv2.imread(path)[:, :, ::-1].astype(np.float32) / 255.0
        img = cv2.resize(img, (self.Tw, self.Th), interpolation=cv2.INTER_AREA)
        # CHW
        img_t = torch.from_numpy(img.transpose(2,0,1)).float()
        return fname, img_t

# 3) Charger vos images de test
IMG_DIR = "dataset_project_iapr2025/test"
files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(".jpg")])
test_ds = TestImageDataset(IMG_DIR, files, target_size=(512,512))
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, num_workers=0)

# 4) Charger le modèle entraîné
model = UNet(len(CLASS_NAMES)).to(DEVICE)
model.load_state_dict(torch.load("best_choco_unet.pth", map_location=DEVICE))
model.eval()

# 5) Boucle d'inférence et collecte des résultats
records = []
with torch.no_grad():
    for fnames, imgs in test_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs)  # (B, C, H, W)
        # somme → counts, clamp, round, to int
        counts = (preds.sum(dim=(2,3))
                       .clamp(min=0)
                       .round()
                       .long()
                       .cpu()
                       .numpy())
        for fname, cnt in zip(fnames, counts):
            # extraire l'ID numérique du nom de fichier
            id_num = int(os.path.splitext(fname)[0])
            row = {"id": id_num}
            for i, cls in enumerate(CLASS_NAMES):
                row[cls] = int(cnt[i])
            records.append(row)

# 6) Création du DataFrame et sauvegarde
df = pd.DataFrame(records)
df = df[["id"] + CLASS_NAMES]  # ordre des colonnes
df.to_csv("submission.csv", index=False)

# 7) Aperçu
print(df.head())